# Piano activity

In [1]:
!pip install ultralytics opencv-python pygame numpy
!python piano_manos_yolo.py best.pt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.7 MB/s eta 0:00:00
python3: can't open file '/content/piano_manos_yolo.py': [Errno 2] No such file or directory


In [3]:
# Entrenamiento de un modelo de 21 puntos de la mano (YOLO pose, fine-tuning)
# Ábrelo en VS Code y ejecuta cada celda (# %%) con el kernel de Colab (GPU T4).
# Si lo prefieres, copia cada celda a un notebook en el navegador.

# %% 1. Entorno
%pip install -U ultralytics
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# %% 2. Drive (para no perder los pesos si Colab se desconecta)
from google.colab import drive
drive.mount("/content/drive")
PROYECTO = "/content/drive/MyDrive/piano_manos"

# %% 3. Fine-tuning desde pesos preentrenados (COCO pose)
# El dataset hand-keypoints (26.768 imgs, 21 puntos por mano) se descarga solo (~369 MB).
from ultralytics import YOLO

model = YOLO("yolo11n-pose.pt")   # también puedes probar "yolo26n-pose.pt" si tu ultralytics es reciente
model.train(
    data="hand-keypoints.yaml",
    epochs=60,
    imgsz=640,
    batch=32,
    patience=15,                  # detiene si no mejora
    project=PROYECTO,
    name="manos_n",
    exist_ok=True,
)

# %% 4. Validación (mira mAP de pose y de caja)
best = f"{PROYECTO}/manos_n/weights/best.pt"
metrics = YOLO(best).val()
print("mAP50-95 pose:", metrics.pose.map)

# %% 5. Prueba rápida con una imagen tuya (sube una foto de tu mano a /content)
# res = YOLO(best)("/content/mi_mano.jpg", save=True)

# %% 6. Descarga best.pt (o cópialo desde Drive a tu PC)
# from google.colab import files; files.download(best)

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=hand-keypoints.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript,

In [ ]:
"""
Piano de manos con TU modelo entrenado (YOLO pose, 21 puntos por mano).

    pip install ultralytics opencv-python pygame numpy
    python piano_manos_yolo.py best.pt

Teclas: Q = salir | M = mostrar/ocultar valores de depuración

Nota: este modelo detecta manos pero NO distingue izquierda/derecha.
Se asigna por posición en pantalla (vista espejo): la mano que queda más a la
izquierda de la imagen = tu mano izquierda. Con una sola mano visible se usa
la mitad de la pantalla en la que esté.
"""

import sys

import cv2
import numpy as np
import pygame
from ultralytics import YOLO

PESOS = sys.argv[1] if len(sys.argv) > 1 else "best.pt"
CAMARA = 0
IMGSZ = 320          # sube a 416/640 si hay GPU o si la detección falla
CONF = 0.5

UMBRAL_PRESIONAR, UMBRAL_SOLTAR = 0.95, 1.15
UMBRAL_PULGAR = (0.65, 0.85)

# Índices de los 21 puntos (muñeca=0, luego 4 por dedo, igual que MediaPipe)
DEDOS = {
    "pulgar": (4, 5), "indice": (8, 5), "medio": (12, 9),
    "anular": (16, 13), "menique": (20, 17),
}
NOTAS = {
    "Left": {"menique": "C4", "anular": "D4", "medio": "E4", "indice": "F4", "pulgar": "G4"},
    "Right": {"pulgar": "A4", "indice": "B4", "medio": "C5", "anular": "D5", "menique": "E5"},
}
FRECUENCIAS = {
    "C4": 261.63, "D4": 293.66, "E4": 329.63, "F4": 349.23, "G4": 392.00,
    "A4": 440.00, "B4": 493.88, "C5": 523.25, "D5": 587.33, "E5": 659.25,
}


def crear_nota(freq, dur=1.2, sr=44100):
    t = np.linspace(0, dur, int(sr * dur), endpoint=False)
    onda = sum(a * np.sin(2 * np.pi * k * freq * t) for k, a in ((1, 1), (2, .5), (3, .25), (4, .12)))
    onda *= np.exp(-3.0 * t) * np.minimum(1, t / 0.005)
    onda = onda / np.max(np.abs(onda)) * 0.6
    return pygame.sndarray.make_sound((onda * 32767).astype(np.int16))


def main():
    pygame.mixer.pre_init(44100, -16, 1, 256)
    pygame.mixer.init()
    pygame.mixer.set_num_channels(16)
    sonidos = {n: crear_nota(f) for n, f in FRECUENCIAS.items()}

    modelo = YOLO(PESOS)
    cap = cv2.VideoCapture(CAMARA)
    if not cap.isOpened():
        raise RuntimeError("No se pudo abrir la cámara.")

    presionado, activas, depurar = {}, {}, True

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        r = modelo.predict(frame, imgsz=IMGSZ, conf=CONF, verbose=False)[0]
        if r.keypoints is not None and len(r.keypoints.xyn) > 0:
            kp = r.keypoints.xyn.cpu().numpy()          # (n, 21, 2) normalizado 0-1
            conf = r.boxes.conf.cpu().numpy()
            idx = list(np.argsort(-conf)[:2])           # máximo 2 manos
            idx.sort(key=lambda i: kp[i, 0, 0])         # ordenar por x de la muñeca
            etiquetas = ["Left", "Right"] if len(idx) == 2 else [
                "Left" if kp[idx[0], 0, 0] < 0.5 else "Right"]

            for i, mano in zip(idx, etiquetas):
                p = kp[i]
                for x, y in p:
                    cv2.circle(frame, (int(x * w), int(y * h)), 3, (0, 200, 255), -1)

                palma = np.hypot(*(p[0] - p[9])) + 1e-6
                for dedo, (punta, base) in DEDOS.items():
                    d = np.hypot(*(p[punta] - p[base])) / palma
                    up, us = UMBRAL_PULGAR if dedo == "pulgar" else (UMBRAL_PRESIONAR, UMBRAL_SOLTAR)
                    clave = (mano, dedo)
                    if not presionado.get(clave, False) and d < up:
                        presionado[clave] = True
                        sonidos[NOTAS[mano][dedo]].play()
                        activas[clave] = 10
                    elif presionado.get(clave, False) and d > us:
                        presionado[clave] = False

                    px, py = int(p[punta][0] * w), int(p[punta][1] * h)
                    if activas.get(clave, 0) > 0:
                        activas[clave] -= 1
                        cv2.circle(frame, (px, py), 18, (0, 255, 0), -1)
                        cv2.putText(frame, NOTAS[mano][dedo], (px - 15, py - 25),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                    if depurar:
                        cv2.putText(frame, f"{d:.2f}", (px - 15, py + 25),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

        cv2.imshow("Piano de manos (YOLO)", frame)
        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"):
            break
        if k == ord("m"):
            depurar = not depurar

    cap.release()
    cv2.destroyAllWindows()
    pygame.quit()


if __name__ == "__main__":
    main()

In [4]:
from google.colab import files
files.download("/content/drive/MyDrive/piano_manos/manos_n/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>